# Phân Tích và Khám Phá Dữ Liệu (EDA) - Dự án Dự báo Nhu cầu Thuê Xe Đạp

Đây là Notebook thực hiện bước đầu tiên của dự án: Phân tích và khám phá dữ liệu (Exploratory Data Analysis - EDA) cho bộ dữ liệu chia sẻ xe đạp theo giờ (UCI Bike Sharing Dataset, file `hour.csv`).

Nội dung chính trong file này bao gồm:
1. Đọc và kiểm tra cấu trúc dữ liệu cơ bản (kiểm tra kiểu dữ liệu, giá trị thiếu).
2. Phân tích phân phối của biến mục tiêu (`cnt`) và giải thích lý do cần biến đổi logarit.
3. Khảo sát nhu cầu thuê xe theo thời gian (theo giờ trong ngày, ngày trong tuần, tháng, ngày làm việc vs ngày nghỉ).
4. Phân tích các yếu tố thời tiết ảnh hưởng đến lượng xe thuê (nhiệt độ, độ ẩm, tốc độ gió).
5. Đánh giá độ tương quan giữa các biến thông qua ma trận tương quan để phát hiện đa cộng tuyến.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Cấu hình hiển thị cho biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

# Đảm bảo thư mục lưu biểu đồ đã tồn tại
os.makedirs("../figures", exist_ok=True)

## 1. Đọc và kiểm tra cấu trúc dữ liệu ban đầu
Chúng ta sẽ load dữ liệu từ file `data/raw/hour.csv` và kiểm tra các thông tin cơ bản: số dòng, số cột, kiểu dữ liệu và xem có giá trị null nào không.

In [ ]:
# Đường dẫn dữ liệu
data_path = "../data/raw/hour.csv"
df = pd.read_csv(data_path)

# Xem 5 dòng đầu tiên
df.head()

In [ ]:
# Kiểm tra số dòng, cột và kiểu dữ liệu
df.info()

In [ ]:
# Kiểm tra giá trị khuyết thiếu (Missing values)
df.isnull().sum()

### Nhận xét ban đầu:
- Dữ liệu có tổng cộng **17,379 dòng** và **17 cột**.
- Các biến hầu hết ở dạng số (`int64` hoặc `float64`), riêng cột `dteday` đang ở dạng chuỗi (object). Lát nữa ở bước tiền xử lý chúng ta sẽ cần ép kiểu sang datetime.
- Bộ dữ liệu rất sạch, **không có giá trị khuyết thiếu (missing value)** nào ở tất cả các cột.

## 2. Phân tích biến mục tiêu (`cnt`)
Biến `cnt` là tổng số lượng xe đạp được thuê trong một giờ. Đây là biến mục tiêu mà nhóm cần dự báo. Hãy vẽ biểu đồ phân phối để xem phân bố của biến này.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Vẽ phân phối gốc của biến cnt
sns.histplot(df['cnt'], kde=True, ax=axes[0], color='royalblue')
axes[0].set_title('Phân phối của số lượng xe thuê (cnt)')
axes[0].set_xlabel('Số lượng xe thuê (cnt)')
axes[0].set_ylabel('Tần suất')

# Vẽ phân phối sau khi log-transform
sns.histplot(np.log1p(df['cnt']), kde=True, ax=axes[1], color='forestgreen')
axes[1].set_title('Phân phối của log1p(cnt)')
axes[1].set_xlabel('log1p(cnt)')
axes[1].set_ylabel('Tần suất')

plt.tight_layout()
# Lưu biểu đồ
plt.savefig("../figures/eda_demand_distribution.png", dpi=150)
plt.show()

# Tính độ lệch (Skewness)
print("Skewness gốc:", df['cnt'].skew())
print("Skewness sau khi log1p:", np.log1p(df['cnt']).skew())

### Nhận xét về biến mục tiêu:
- Phân phối gốc của `cnt` bị **lệch phải rất nhiều** (Skewness khoảng 1.27). Phần lớn các giờ có lượng thuê xe thấp (dưới 200 xe/giờ), trong khi có một số ít giờ cao điểm lượng xe lên tới gần 1000 xe.
- Khi áp dụng phép biến đổi logarit $y = \log(1 + cnt)$, phân phối trở nên cân đối hơn, gần với phân phối chuẩn hơn nhiều (Skewness giảm xuống còn khoảng -0.8).
- **Đề xuất**: Trong quá trình huấn luyện các mô hình học máy (như Linear Regression hay Neural Networks), việc dự báo trên biến mục tiêu đã biến đổi logarit thường sẽ giúp mô hình hội tụ tốt hơn và giảm tác động từ các điểm dữ liệu quá lớn (outliers).

## 3. Khảo sát nhu cầu thuê xe theo thời gian
Chúng ta sẽ phân tích xem lượng xe thuê thay đổi thế nào theo các biến thời gian: giờ trong ngày (`hr`), ngày trong tuần (`weekday`), mùa (`season`), năm (`yr`), và ngày làm việc vs ngày nghỉ (`workingday`).

### 3.1. Nhu cầu thuê xe theo giờ trong ngày (hr)
Nhu cầu thuê xe chắc chắn sẽ biến động mạnh giữa các giờ trong ngày. Chúng ta hãy so sánh biến động này giữa ngày làm việc (`workingday = 1`) và ngày nghỉ/cuối tuần (`workingday = 0`).

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='hr', y='cnt', hue='workingday', errorbar='ci', palette='Set1', marker='o')
plt.title('Nhu cầu thuê xe trung bình theo giờ (Ngày làm việc vs Ngày nghỉ/Lễ)')
plt.xlabel('Giờ trong ngày (hr)')
plt.ylabel('Số lượng xe thuê trung bình')
plt.legend(title='Ngày làm việc', labels=['Ngày nghỉ/Lễ (0)', 'Ngày làm việc (1)'])
plt.xticks(range(0, 24))
plt.tight_layout()
plt.savefig("../figures/eda_demand_by_hour.png", dpi=150)
plt.show()

### Nhận xét:
- Biểu đồ cho thấy sự khác biệt rõ rệt về hành vi thuê xe giữa ngày đi làm và ngày nghỉ:
  - **Ngày làm việc (đường màu xanh)**: Có hai đỉnh rất rõ ràng vào khung giờ đi làm/tan tầm là **8 giờ sáng** và **17-18 giờ chiều**. Đây là lúc người dân thuê xe đạp để di chuyển đi học, đi làm.
  - **Ngày nghỉ/Lễ (đường màu đỏ)**: Nhu cầu thuê xe tăng dần từ sáng và đạt đỉnh vào giữa ngày từ **12 giờ đến 16 giờ chiều**. Hành vi này phù hợp với nhu cầu vui chơi giải trí, dã ngoại cuối tuần.
- Điều này chứng tỏ biến tương tác giữa `hr` và `workingday` sẽ là một đặc trưng cực kỳ quan trọng cho các mô hình học máy sau này.

### 3.2. Nhu cầu thuê xe theo ngày trong tuần (weekday)
Chúng ta hãy xem phân bố lượng xe thuê theo từng ngày trong tuần (từ Chủ Nhật đến Thứ Bảy).

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='weekday', y='cnt', palette='Set3')
plt.title('Phân bố nhu cầu thuê xe theo ngày trong tuần')
plt.xlabel('Ngày trong tuần (0: Chủ Nhật, ..., 6: Thứ Bảy)')
plt.ylabel('Số lượng xe thuê (cnt)')
plt.xticks(ticks=range(7), labels=['CN', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7'])
plt.tight_layout()
plt.savefig("../figures/eda_demand_by_weekday.png", dpi=150)
plt.show()

### Nhận xét:
- Nhìn chung, mức phân bố lượng xe thuê ở các ngày trong tuần tương đối đồng đều.
- Tuy nhiên, ta thấy số lượng các điểm outlier (những giờ có lượng xe thuê đột biến cao) xuất hiện nhiều hơn ở các ngày trong tuần (Thứ Hai đến Thứ Sáu) so với cuối tuần, tương thích với nhu cầu giờ cao điểm lớn trong ngày đi làm.

### 3.3. Nhu cầu thuê xe theo mùa (`season`) và tình trạng thời tiết (`weathersit`)
Chúng ta phân tích nhu cầu thuê xe dưới tác động của mùa vụ và thời tiết.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Vẽ boxplot theo mùa
sns.boxplot(data=df, x='season', y='cnt', ax=axes[0], palette='coolwarm')
axes[0].set_title('Nhu cầu thuê xe theo mùa')
axes[0].set_xlabel('Mùa')
axes[0].set_ylabel('Số lượng xe thuê (cnt)')
axes[0].set_xticklabels(['Xuân', 'Hạ', 'Thu', 'Đông'])

# Vẽ boxplot theo tình trạng thời tiết
sns.boxplot(data=df, x='weathersit', y='cnt', ax=axes[1], palette='viridis')
axes[1].set_title('Nhu cầu thuê xe theo tình trạng thời tiết')
axes[1].set_xlabel('Thời tiết (1: Đẹp -> 4: Xấu)')
axes[1].set_ylabel('Số lượng xe thuê (cnt)')
axes[1].set_xticklabels(['Đẹp/Ít mây', 'Sương mù/Mây', 'Mưa/Tuyết nhẹ', 'Mưa to/Tuyết to'])

plt.tight_layout()
plt.savefig("../figures/eda_demand_by_season.png", dpi=150)
plt.show()

### Nhận xét:
- **Theo mùa**: Lượng xe thuê thấp nhất vào mùa Xuân (do thời tiết lạnh giá đầu năm). Nhu cầu tăng mạnh nhất vào mùa Hạ (mùa 2) và mùa Thu (mùa 3), sau đó giảm nhẹ vào mùa Đông khi thời tiết lạnh trở lại.
- **Theo thời tiết**: Thời tiết ảnh hưởng trực tiếp đến quyết định thuê xe đạp. Lượng thuê xe cao nhất khi trời đẹp, ít mây (`weathersit = 1`). Khi thời tiết xấu hơn (sương mù, mưa rào, tuyết rơi), nhu cầu giảm hẳn. Đặc biệt khi trời mưa to hoặc tuyết dày (`weathersit = 4`), lượng thuê xe gần như bằng không.

## 4. Phân tích mối tương quan giữa các biến số
Chúng ta sẽ tính toán ma trận tương quan Pearson để đánh giá mối liên hệ tuyến tính giữa các biến số liên tục (nhiệt độ, độ ẩm, tốc độ gió) và biến mục tiêu `cnt`.

In [ ]:
# Chọn các cột số quan trọng để vẽ ma trận tương quan
cols_to_corr = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt']
corr_matrix = df[cols_to_corr].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, square=True)
plt.title('Ma trận tương quan giữa các biến số')
plt.tight_layout()
plt.savefig("../figures/eda_correlation_matrix.png", dpi=150)
plt.show()

### Nhận xét quan trọng về tương quan:
1. **Mối tương quan với biến mục tiêu `cnt`**:
   - Biến `temp` (nhiệt độ) và `atemp` (nhiệt độ cảm nhận) có tương quan dương mạnh nhất với `cnt` (khoảng 0.40). Điều này cho thấy khi trời ấm lên, nhu cầu thuê xe đạp tăng cao.
   - Biến `hr` (giờ) có tương quan dương ở mức 0.39, phản ánh sự biến động mạnh của lượng xe thuê theo các khung giờ trong ngày.
   - Biến `hum` (độ ẩm) có tương quan âm (-0.32), tức là độ ẩm càng cao (trời âm u hoặc oi bức) thì người dân càng ít thuê xe đạp.
   - Biến `weathersit` có tương quan âm (-0.14), chứng tỏ thời tiết xấu làm giảm lượng thuê xe.

2. **Hiện tượng đa cộng tuyến (Multicollinearity)**:
   - Hai biến `temp` (nhiệt độ thực tế) và `atemp` (nhiệt độ cảm nhận) có tương quan tuyến tính gần như tuyệt đối (**0.99**).
   - Việc giữ lại cả 2 biến này trong các mô hình hồi quy tuyến tính có thể gây ra hiện tượng đa cộng tuyến, làm sai lệch các hệ số ước lượng của mô hình.
   - **Đề xuất**: Thành viên phụ trách làm feature engineering (Thành viên 2) nên cân nhắc loại bỏ một trong hai biến này (ví dụ chỉ giữ lại `temp`) hoặc tạo một đặc trưng kết hợp để tránh trùng lặp thông tin.

## 5. Kết luận chung từ quá trình EDA
Qua các phân tích trên, Thành viên 1 rút ra một số kết luận định hướng cho các bước tiếp theo của nhóm:
1. **Biến mục tiêu**: Nên áp dụng log-transform (`log1p`) khi huấn luyện các mô hình để tránh bị ảnh hưởng bởi độ lệch lớn của dữ liệu thực tế.
2. **Feature Engineering (Thành viên 2)**:
   - Cần khai thác các biến tương tác quan trọng, đặc biệt là sự kết hợp giữa giờ (`hr`) và ngày làm việc (`workingday`).
   - Nên loại bỏ biến `atemp` hoặc xử lý đa cộng tuyến vì nó bị trùng lặp thông tin với `temp` (tương quan 0.99).
   - Các biến phân loại như `season`, `weathersit`, `weekday` nên được mã hóa phù hợp (ví dụ: One-Hot Encoding hoặc tạo các biến tuần hoàn dạng Fourier features).
3. **Huấn luyện mô hình (Thành viên 3)**:
   - Nhu cầu thuê xe có tính chu kỳ rất rõ rệt theo ngày (24 giờ) và theo tuần (7 ngày). Các mô hình chuỗi thời gian như LSTM, GRU hoặc các mô hình học máy có tạo biến trễ (lag features) sẽ khai thác được rất tốt các tính chất tuần hoàn này.